# 04 – Rolling-origin validation and sensitivity analysis
Notebook 03 used a single train/test split, on which the three ML models were within ≈ 1% of each other. This notebook checks:

1. **Rolling-origin validation:** do the models beat the baselines *consistently* across many test periods, and which model should be chosen?
2. **Sensitivity analysis:** how much do the results depend on the main modelling choices (tree depth, `max_hold`, amount of training history, occupancy threshold)?

⏱ Section 1 takes roughly 5–10 minutes (it trains about 200 models; most of the time is the MLP). Section 2 takes a few more minutes.

In [ ]:
import sys
sys.path.insert(0, "../src")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from occforecast.load import load_occupancy
from occforecast.resample import resample_all
from occforecast.baselines import weekly_profile, persistence
from occforecast.evaluate import score, rolling_origin
from occforecast.model import make_features, make_model, fit_predict

DATA = "../data/raw/5occupancySensor_MayToDec2024_9MRows.csv"
SPLIT = pd.Timestamp("2025-07-01", tz="Australia/Melbourne")
TZ = "Australia/Melbourne"

events = load_occupancy(DATA)
labels = {s: f"Space {chr(65 + i)}" for i, s in enumerate(sorted(events["space"].unique()))}

def space_series(max_hold="48h"):
    """Resample all spaces and return {label: headcount series}."""
    s = resample_all(events, freq="15min", max_hold=max_hold).sort_index()
    return {labels[sp]: g["headcount_mean"] for sp, g in s.groupby("space")}

ys = space_series()
NAMES = {"linear": "Linear regression", "tree": "Decision tree", "mlp": "Neural network (MLP)"}

## 1. Rolling-origin validation
Ten test windows of 8 weeks each, starting every two months from Sep 2024 to Mar 2026. For each window, every method is trained on **all data before the window starts** (expanding window) and scored on the window. All methods are scored on the same intervals.

**Model-selection rule (fixed before looking at the results):** choose the ML model with the lowest mean MAE over all windows and spaces. If two models are within 2% of each other, choose the simpler and faster one (linear < tree < MLP), because the extra complexity is not justified by accuracy.

In [ ]:
origins = list(pd.date_range("2024-09-01", "2026-03-01", freq="2MS", tz=TZ))
print("Origins:", [o.strftime("%Y-%m-%d") for o in origins])

all_rows = []
t_start = time.perf_counter()
for name, y in ys.items():
    X = make_features(y)
    forecasters = {
        "Weekly profile": lambda s, o: weekly_profile(s[s.index < o], s.index[s.index >= o]),
        "Persistence (7 days)": lambda s, o: persistence(s, s.index[s.index >= o], lag="7D"),
    }
    for m, label in NAMES.items():
        forecasters[label] = lambda s, o, m=m, X=X: fit_predict(s, o, make_model(m), features=X)
    res = rolling_origin(y, origins, "56D", forecasters)
    res["space"] = name
    all_rows.append(res)
    print(f"{name}: {res['origin'].nunique()} windows  ({time.perf_counter() - t_start:.0f} s elapsed)")

ro = pd.concat(all_rows, ignore_index=True)

### 1a. Average over all windows and spaces

In [ ]:
order = ["Weekly profile", "Persistence (7 days)"] + list(NAMES.values())
summary = ro.groupby("method")[["MAE", "RMSE", "F1"]].agg(["mean", "std"]).loc[order]
display(summary.round(3))

# How often does each ML model beat the best baseline (on MAE) in the same window?
w = ro.pivot_table(index=["space", "origin"], columns="method", values="MAE")
best_base = w[["Weekly profile", "Persistence (7 days)"]].min(axis=1)
win_rate = {m: (w[m] < best_base).mean() * 100 for m in NAMES.values()}
print("Windows where model beats the best baseline (MAE):",
      {k: f"{v:.0f}%" for k, v in win_rate.items()}, f"(of {len(w)} space-windows)")

### 1b. Error over time

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for m in order:
    s = ro[ro.method == m].groupby("origin")["MAE"].mean()
    ax.plot(s.index, s.values, marker="o", label=m, lw=2 if m in NAMES.values() else 1,
            ls="-" if m in NAMES.values() else "--")
ax.set_ylabel("MAE (people), mean over spaces")
ax.set_xlabel("Start of 8-week test window")
ax.set_title("Rolling-origin validation")
ax.legend(fontsize=8); plt.show()

### 1c. Model selection

In [ ]:
ml = ro[ro.method.isin(NAMES.values())].groupby("method")["MAE"].mean().sort_values()
best, best_mae = ml.index[0], ml.iloc[0]
complexity = {"Linear regression": 0, "Decision tree": 1, "Neural network (MLP)": 2}
near = [m for m in ml.index if ml[m] <= best_mae * 1.02]
chosen = min(near, key=complexity.get)
print(ml.round(4).to_string())
print(f"\nLowest mean MAE: {best}. Within 2%: {near}. Chosen (simplest of those): {chosen}")

**Findings:** _TODO – do the ML models beat the baselines consistently (win rate)? Which model is chosen and why? Which periods are hardest (e.g. windows covering the summer shutdown or the start of semester)?_

## 2. Sensitivity analysis
Each test changes **one** choice and keeps everything else at its default, using the single split from notebook 03 (train before 1 Jul 2025, test after). Results are averaged over spaces A, B, D, E.

### 2a. Decision-tree depth (overfitting)
A deeper tree fits the training data more closely. If test error starts rising while training error keeps falling, the tree is overfitting (memorising noise rather than learning the pattern). Two settings of the minimum leaf size are compared: the default (20 samples per leaf), which limits how finely the tree can split, and 1, which lets a deep tree memorise individual intervals.

In [ ]:
depths = [2, 3, 4, 6, 8, 10, 12, 15, 20, None]
rows = []
for name, y in ys.items():
    if y[y.index >= SPLIT].dropna().empty:
        continue
    X = make_features(y)
    train = (y.index < SPLIT) & y.notna().to_numpy()
    test = (y.index >= SPLIT) & y.notna().to_numpy()
    for leaf in [20, 1]:
        for d in depths:
            model = make_model("tree", max_depth=d, min_samples_leaf=leaf).fit(X[train], y[train])
            rows.append({"space": name, "leaf": leaf, "depth": d if d else 40,
                         "Train MAE": score(y[train], model.predict(X[train]))["MAE"],
                         "Test MAE": score(y[test], np.clip(model.predict(X[test]), 0, None))["MAE"]})
depth = pd.DataFrame(rows).groupby(["leaf", "depth"])[["Train MAE", "Test MAE"]].mean()

fig, ax = plt.subplots(figsize=(8, 4))
for leaf, style in [(20, "-"), (1, ":")]:
    d = depth.loc[leaf]
    ax.plot(d.index, d["Train MAE"], marker="o", ls=style, color="C0", label=f"Training (min leaf {leaf})")
    ax.plot(d.index, d["Test MAE"], marker="o", ls=style, color="C1", label=f"Test (min leaf {leaf})")
ax.set_xticks(depth.loc[20].index, [str(d) if d < 40 else "None" for d in depth.loc[20].index])
ax.set_xlabel("max_depth"); ax.set_ylabel("MAE (people)")
ax.set_title("Decision tree: training vs test error"); ax.legend(fontsize=8); plt.show()
display(depth.unstack("leaf").round(3))

**Findings:** _TODO – which depth gives the lowest test error? With min leaf 1, from what depth does the tree overfit (training error keeps falling, test error rises)? Does the default (depth 10, min leaf 20) sit near the best test error?_

### 2b. `max_hold` (how long a reading is carried forward)
Changes how silences are handled during resampling. This changes both the training data and the test "truth", so the number of scored intervals is shown too.

In [ ]:
rows = []
for hold in ["12h", "24h", "48h", None]:
    ys_h = space_series(max_hold=hold)
    for name, y in ys_h.items():
        test = y[y.index >= SPLIT].dropna()
        if test.empty:
            continue
        X = make_features(y)
        preds = {"Weekly profile": weekly_profile(y[y.index < SPLIT], test.index),
                 "Linear regression": fit_predict(y, SPLIT, make_model("linear"), features=X),
                 "Decision tree": fit_predict(y, SPLIT, make_model("tree"), features=X)}
        for m, p in preds.items():
            rows.append({"max_hold": hold or "unlimited", "space": name, "method": m,
                         **score(test, p.reindex(test.index))})
hold = pd.DataFrame(rows)
display(hold.pivot_table(index="max_hold", columns="method", values="MAE").round(3)
        .loc[["12h", "24h", "48h", "unlimited"]])
n_scored = hold[hold.method == "Weekly profile"].groupby("max_hold")["n"].sum()
display(n_scored.loc[["12h", "24h", "48h", "unlimited"]].rename("test intervals scored (all spaces)"))

**Findings:** _TODO – does the choice of `max_hold` change the ranking of methods or their errors noticeably? If not, the 48 h choice from notebook 01 is not driving the conclusions._

### 2c. Amount of training history
Models are trained only on the last 3, 6 or 12 months before the split, or on all available history. Features (lags) are unchanged; only the training rows are restricted.

In [ ]:
rows = []
for months in [3, 6, 12, None]:
    for name, y in ys.items():
        test = y[y.index >= SPLIT].dropna()
        if test.empty:
            continue
        X = make_features(y)
        y_train = y.copy()
        if months:
            y_train[y_train.index < SPLIT - pd.DateOffset(months=months)] = np.nan
        n_train = int(y_train[y_train.index < SPLIT].notna().sum())
        for m in ["linear", "tree"]:
            p = fit_predict(y_train, SPLIT, make_model(m), features=X).reindex(test.index)
            rows.append({"history": f"{months} months" if months else "all", "space": name,
                         "method": NAMES[m], "n_train": n_train, **score(test, p)})
hist = pd.DataFrame(rows)
display(hist.pivot_table(index="history", columns="method", values="MAE")
        .loc[["3 months", "6 months", "12 months", "all"]].round(3))

**Findings:** _TODO – how much history is needed before accuracy stops improving? (Practical meaning: how long a new sensor must run before the model is useful.)_

### 2d. Occupancy threshold in F1
F1 depends on the headcount at which an interval counts as "occupied" (default 0.5). This checks that the F1 comparison does not depend on that choice.

In [ ]:
rows = []
for name, y in ys.items():
    test = y[y.index >= SPLIT].dropna()
    if test.empty:
        continue
    X = make_features(y)
    preds = {"Weekly profile": weekly_profile(y[y.index < SPLIT], test.index),
             "Linear regression": fit_predict(y, SPLIT, make_model("linear"), features=X).reindex(test.index),
             "Decision tree": fit_predict(y, SPLIT, make_model("tree"), features=X).reindex(test.index)}
    for thr in [0.25, 0.5, 1.0, 2.0]:
        for m, p in preds.items():
            rows.append({"threshold": thr, "space": name, "method": m,
                         "F1": score(test, p, threshold=thr)["F1"]})
thr = pd.DataFrame(rows).pivot_table(index="threshold", columns="method", values="F1")
display(thr.round(3))

**Findings:** _TODO – do the models keep their F1 advantage at every threshold?_

## 3. Summary of sensitivity
_TODO – one short paragraph: which choices matter most, which barely matter, and therefore how robust the conclusions are._